# Day 10: Gridding: blocks, numbers, neighbours (and the Week 2 review)

**Time:** about 1 hour  |  **Needs:** Days 1 to 9
**Reservoir idea:** dividing the reservoir into blocks, numbering them, finding each block's neighbours, and treating the outer edge as a closed (no-flow) boundary
**Python you will learn:** 2-D and 3-D arrays, `reshape`, indexing with `a[k, j, i]`, `axis=` in sums and means, integer division `//` and remainder `%`, turning a flat block number into `(i, j, k)` and back

By the end of today you will be able to:

1. number the blocks of a 1-D, 2-D or 3-D grid the way the thesis and `kusim` do,
2. view a flat array as layers, rows and columns with `reshape`, and average over layers,
3. convert a block number to `(i, j, k)` and back, for all blocks at once,
4. build the six neighbour arrays and the "is there a neighbour?" masks of a grid,
5. assemble all of this into your own grid, and compare it with `kusim`'s `Grid`.

**This is the last day of Week 2.** Section 8 is a review quiz of Days 6 to 10.

## 0. Setup and recap

**Recap of Days 8 and 9.** Slices cut arrays (`a[1:]`, `a[::-1]`), masks select items by a condition, `np.where` is an `if` for a whole array. `np.diff` gives differences between neighbours, and Darcy's law gives the flow between two blocks from their pressure difference. Today we study *which* blocks are neighbours.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DARCY_UNITS = 0.001127      # Darcy's law in oilfield units (Day 9)
print("NumPy", np.__version__, "| ready")

## 1. Story: where is block 2000?

*(An invented but typical situation.)* A reservoir model has 20 blocks along $x$, 20 along $y$ and 5 layers. The input says "producer in block 2000". Is that the top of the reservoir, the middle, or the bottom? And which corner?

The simulator does not know about "corners" or "layers". It only has a **list of 2000 blocks, numbered 1 to 2000**. The engineer must know the numbering rule to see that block 2000 is the *last* block of the *bottom* layer. A well in the wrong block gives a perfectly valid simulation of the wrong reservoir, and nothing crashes to warn you. Today we learn the numbering rule and the tools to move between block numbers and positions.

## 2. Numbering the blocks

### One dimension

A row of $n_x$ blocks is numbered $1,2,\dots,n_x$ from left to right. The pressure of all blocks is one array of length $n_x$.

**The off-by-one trap.** Block numbers start at **1** (the way engineers count), but Python indices start at **0**. Block 1 is `p[0]`, block `n` is `p[n - 1]`. `kusim` keeps user-facing block numbers from 1 and converts inside.

### Two dimensions

For $n_x$ blocks along $x$ and $n_y$ along $y$, the thesis numbers the blocks **along $x$ first**, then moves to the next row:

```
 y=3 |  9  10  11  12
 y=2 |  5   6   7   8
 y=1 |  1   2   3   4
      x=1  2   3   4
```

The block number of position $(i, j)$ (each counted from 1) is

$$\text{block} = (j-1)\,n_x + i$$

### Three dimensions

With $n_z$ layers, the numbering continues layer by layer, **top layer first** (so depth increases with the layer number $k$):

$$\text{block} = (k-1)\,n_x n_y + (j-1)\,n_x + i \tag{1}$$

Equation (1) is in the docstring of `kusim/grid.py`. The first layer has the blocks $1$ to $n_x n_y$, the second layer the next $n_x n_y$, and so on.

For the 20 × 20 × 5 grid of the story: $(5-1)\cdot400 + (20-1)\cdot20 + 20 = 1600+380+20 = 2000$. Block 2000 is $(i,j,k)=(20,20,5)$: the far corner of the **bottom** layer.

**Your turn 2.1.** Use equation (1). In a grid with `nx = 4`, `ny = 3`, `nz = 2` calculate `b_a`, the block number of $(i,j,k)=(3,2,2)$, and `b_b`, the block number of $(1,1,1)$. Then in the 20 × 20 × 5 grid of the story, `b_c`, the block number of $(1,1,5)$.

In [ ]:
nx, ny, nz = 4, 3, 2

# YOUR CODE HERE
b_a = None
b_b = None
b_c = None

In [ ]:
# Check (do not edit)
assert b_a == 19, f"(3,2,2) in a 4x3x2 grid is block 19, got {b_a}"
assert b_b == 1, f"(1,1,1) is always block 1, got {b_b}"
assert b_c == 1601, f"(1,1,5) in the 20x20x5 grid is block 1601, got {b_c}"
print("Correct")

<details><summary>Solution</summary>

```python
b_a = (2 - 1) * nx * ny + (2 - 1) * nx + 3
b_b = (1 - 1) * nx * ny + (1 - 1) * nx + 1
b_c = (5 - 1) * 20 * 20 + (1 - 1) * 20 + 1
```

</details>

## 3. 2-D and 3-D arrays, and `reshape`

`kusim` keeps every cell property (pressure, saturation, permeability, ...) as **one flat array**, in block-number order. But a person wants to see layers and maps. `reshape` shows the *same numbers* as a table or a stack of tables, without copying them.

For `nz = 2` layers of `ny = 3` rows and `nx = 4` columns, the shape is written **`(nz, ny, nx)`**: the slowest-changing index first, the fastest last.

In [ ]:
nx, ny, nz = 4, 3, 2
blocks = np.arange(1, nx * ny * nz + 1)      # block numbers 1 ... 24
b3d = blocks.reshape(nz, ny, nx)

print(blocks)
print(b3d.shape)
print(b3d)

The output shows two tables (layers), each with 3 rows and 4 columns. In the first, the first row is `1 2 3 4`, the second row `5 6 7 8`. This is the numbering diagram of Section 2, now made by NumPy.

**Indexing a 3-D array** uses one index for each direction, **in the order (layer, row, column)**, and counts from 0:

| You write | You get |
|---|---|
| `b3d[0]` | the whole top layer (a 3 × 4 table) |
| `b3d[1, 2, 3]` | layer 2, row 3, column 4 (all counted from 1): block 24 |
| `b3d[:, 0, :]` | the first row of **every** layer (a vertical cross-section) |
| `b3d[:, :, 0]` | the first column of every layer |
| `b3d.ravel()` | back to a flat array |

**Predict.** What do `b3d[0, 1, 2]`, `b3d[1, 0, 0]` and `b3d[:, 2, 3]` give? Write your guesses, then run.

In [ ]:
print(b3d[0, 1, 2])
print(b3d[1, 0, 0])
print(b3d[:, 2, 3])

`b3d[0, 1, 2]` is layer 1, row 2, column 3 (counting from 1): block 7. `b3d[1, 0, 0]` is the first block of the second layer: 13. `b3d[:, 2, 3]` is the vertical column at the last row and column: blocks 12 and 24, one per layer.

Here is the picture, with the numbers written on a colour map of each layer (a loop writes each number):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for k in range(nz):
    ax = axes[k]
    ax.imshow(np.zeros((ny, nx)), cmap="Blues", vmin=0, vmax=1, origin="lower")
    for j in range(ny):
        for i in range(nx):
            ax.text(i, j, str(b3d[k, j, i]), ha="center", va="center", fontsize=12)
    ax.set_title(f"layer {k + 1}" + (" (top)" if k == 0 else " (below)"))
    ax.set_xticks(range(nx), [str(i + 1) for i in range(nx)])
    ax.set_yticks(range(ny), [str(j + 1) for j in range(ny)])
    ax.set_xlabel("$i$ (along $x$)")
    ax.set_ylabel("$j$ (along $y$)")
plt.tight_layout()
plt.show()

**Two ways to draw the same table.** A printed NumPy table shows row 1 (`1 2 3 4`) at the **top**; the map above puts $j=1$ at the **bottom** (`origin="lower"`), like a map with $y$ increasing upwards. The numbers are the same; only the drawing differs. Be careful when you compare a printed array with a map.

### Whole-layer properties

A model with layers of different rock gives each layer its own value. Because blocks are numbered layer by layer, **`np.repeat`** does it neatly: `np.repeat([a, b], m)` gives `a` repeated `m` times, then `b` repeated `m` times.

**Your turn 3.1.** The two layers of our 4 × 3 × 2 grid have permeabilities of 200 md (top) and 50 md (bottom). Build the flat array `perm_flat` of length 24 with `np.repeat` (each layer has `nx * ny` blocks). Then store `perm_3d` (the same numbers reshaped to `(nz, ny, nx)`) and `k_block_19` (the permeability of block 19, remembering that its array index is 18).

In [ ]:
layer_perm = np.array([200.0, 50.0])

# YOUR CODE HERE
perm_flat = None
perm_3d = None
k_block_19 = None

In [ ]:
# Check (do not edit)
assert perm_flat.shape == (24,), f"perm_flat should have 24 items, got shape {perm_flat.shape}"
assert perm_flat[:12].min() == 200 and perm_flat[12:].max() == 50, "the first 12 blocks are layer 1, the next 12 are layer 2"
assert perm_3d.shape == (2, 3, 4) and perm_3d[1, 1, 2] == 50, "perm_3d should have shape (2, 3, 4)"
assert k_block_19 == 50, f"block 19 lies in layer 2 (50 md), got {k_block_19}"
print("Correct: block 19 is in the bottom layer")

<details><summary>Solution</summary>

```python
perm_flat = np.repeat(layer_perm, nx * ny)
perm_3d = perm_flat.reshape(nz, ny, nx)
k_block_19 = perm_flat[18]
```
Reading `perm_flat[18]` for block 19 is the off-by-one rule of Section 2: array index = block number - 1.
</details>

### Averages over layers: `axis`

`.mean()` and `.sum()` can work along one direction of a 3-D array, using **`axis`**: `axis=0` goes through the layers, `axis=1` through the rows, `axis=2` through the columns. The direction you name **disappears** from the result. You can also name several at once: `axis=(1, 2)` averages over each whole layer.

| You write | Shape of the result | Meaning |
|---|---|---|
| `a.mean()` | a single number | the average of everything |
| `a.mean(axis=0)` | `(ny, nx)` | for each map position, the average over the layers (a map) |
| `a.mean(axis=(1, 2))` | `(nz,)` | the average of each layer (one number per layer) |

**Your turn 3.2.** The water saturation of the 24 blocks (flat, in block order) is `sw24` below. Reshape it to `sw3d` of shape `(nz, ny, nx)`. Store `layer_mean` (the average saturation of each of the two layers), `map_mean` (the average over the layers at each map position, a `(3, 4)` array) and `top_layer` (the top layer as a `(3, 4)` array).

In [ ]:
sw24 = np.linspace(0.2, 0.8, 24)

# YOUR CODE HERE
sw3d = None
layer_mean = None
map_mean = None
top_layer = None

In [ ]:
# Check (do not edit)
assert sw3d.shape == (2, 3, 4), f"sw3d shape should be (2, 3, 4), got {sw3d.shape}"
assert np.allclose(layer_mean, [0.34348, 0.65652], atol=1e-4), f"layer_mean is {layer_mean}"
assert map_mean.shape == (3, 4) and top_layer.shape == (3, 4), "both should be 3 x 4 tables"
assert abs(map_mean[0, 0] - (sw24[0] + sw24[12]) / 2) < 1e-12, "map_mean[0, 0] averages blocks 1 and 13"
print("Correct: the top layer is drier (lower Sw) than the bottom one:", np.round(layer_mean, 3))

<details><summary>Solution</summary>

```python
sw3d = sw24.reshape(nz, ny, nx)
layer_mean = sw3d.mean(axis=(1, 2))
map_mean = sw3d.mean(axis=0)
top_layer = sw3d[0]
```
This is how the `viz` module draws a layer map (`to_3d(...)[k]`) or a vertical average, and how a layer-by-layer water saturation is reported.
</details>

## 4. From a flat index to `(i, j, k)`, and back

A simulator works on flat arrays, so for every block it needs to know *where* the block is. Going from the **array index** $n=0,1,\dots$ (block number minus 1) to the position uses two operations you know from school division:

- `//` is **integer division** (the quotient, rounded down): `17 // 5` is `3`,
- `%` is the **remainder**: `17 % 5` is `2`.

Think of it as clock arithmetic: after `nx` blocks you go to the next row, after `nx * ny` blocks to the next layer.

$$i_0 = n \bmod n_x,\qquad j_0 = \left\lfloor \frac{n}{n_x}\right\rfloor \bmod n_y,\qquad k_0 = \left\lfloor \frac{n}{n_x n_y}\right\rfloor \tag{2}$$

(Here $i_0,j_0,k_0$ count from 0, the Python way.) Both `//` and `%` work on **whole arrays**, so one call finds the position of all blocks.

In [ ]:
print(17 // 5, 17 % 5)
n = np.arange(24)
print(n % 4)                 # i0 for every block of a grid with nx = 4
print((n // 4) % 3)          # j0
print(n // (4 * 3))          # k0

**Your turn 4.1.** Write `cell_positions(nx, ny, nz)` that returns three arrays `i, j, k` (**counted from 0**), each with one value per block, in block order, using equations (2).

In [ ]:
# YOUR CODE HERE
def cell_positions(nx, ny, nz):
    """Zero-based positions (i, j, k) of every block, in block order (x fastest, then y, then z)."""
    pass

In [ ]:
# Check (do not edit)
i4, j4, k4 = cell_positions(4, 3, 2)
assert len(i4) == 24, "one position per block"
assert list(i4[:6]) == [0, 1, 2, 3, 0, 1], f"i starts 0 1 2 3 0 1, got {list(i4[:6])}"
assert list(j4[:6]) == [0, 0, 0, 0, 1, 1], f"j starts 0 0 0 0 1 1, got {list(j4[:6])}"
assert list(k4[[0, 11, 12, 23]]) == [0, 0, 1, 1], "k changes after the 12th block"
assert (i4[18], j4[18], k4[18]) == (2, 1, 1), "index 18 (block 19) is (i, j, k) = (3, 2, 2) from 1"
print("Correct")

<details><summary>Solution</summary>

```python
def cell_positions(nx, ny, nz):
    """Zero-based positions (i, j, k) of every block, in block order (x fastest, then y, then z)."""
    n = np.arange(nx * ny * nz)
    i = n % nx
    j = (n // nx) % ny
    k = n // (nx * ny)
    return i, j, k
```
`kusim` makes the same arrays with `np.meshgrid(...)` followed by `.ravel()`; the result is identical. Ours shows the arithmetic.
</details>

**Your turn 4.2.** Write `block_number(i, j, k, nx, ny)` for positions counted **from 1** (equation (1)) that works for single numbers **and arrays**. Then check that your two functions are the inverse of each other: store `roundtrip_ok`, which is `True` if applying `block_number` to the positions from `cell_positions` (adding 1 to each) gives the numbers 1 to 24 in order. Use `np.array_equal(a, b)`.

In [ ]:
i4, j4, k4 = cell_positions(4, 3, 2)

# YOUR CODE HERE
def block_number(i, j, k, nx, ny):
    """Thesis block number (from 1) of the position (i, j, k) (each from 1)."""
    pass

roundtrip_ok = None

In [ ]:
# Check (do not edit)
assert block_number(3, 2, 2, 4, 3) == 19, "(3, 2, 2) is block 19"
assert block_number(20, 20, 5, 20, 20) == 2000, "the last block of the 20 x 20 x 5 grid is 2000"
assert roundtrip_ok, "going to (i, j, k) and back should give the block numbers 1 ... 24"
print("Correct: block numbers and positions convert both ways")

<details><summary>Solution</summary>

```python
def block_number(i, j, k, nx, ny):
    """Thesis block number (from 1) of the position (i, j, k) (each from 1)."""
    return (k - 1) * nx * ny + (j - 1) * nx + i

roundtrip_ok = np.array_equal(block_number(i4 + 1, j4 + 1, k4 + 1, 4, 3), np.arange(1, 25))
```

</details>

## 5. Neighbours and closed boundaries

The flow between blocks (Day 9) needs to know, for every block, **who its neighbours are**. In the flat numbering (array indices from 0):

| Direction | Neighbour's index | Exists unless... |
|---|---|---|
| `x_minus` | $n-1$ | $i_0 = 0$ (first column) |
| `x_plus` | $n+1$ | $i_0 = n_x-1$ (last column) |
| `y_minus` | $n-n_x$ | $j_0 = 0$ (first row) |
| `y_plus` | $n+n_x$ | $j_0 = n_y-1$ (last row) |
| `z_minus` | $n-n_xn_y$ | $k_0 = 0$ (top layer) |
| `z_plus` | $n+n_xn_y$ | $k_0 = n_z-1$ (bottom layer) |

For example, in the 4 × 3 × 2 grid block 6 (index 5, $(i,j,k)=(2,2,1)$) has its neighbours in blocks 5, 7 (along $x$), 2, 10 (along $y$) and 18 (below). It has none above: it is in the top layer.

**What about the edge?** A block on the edge has no neighbour in that direction. Instead of "no value", which would complicate every formula, `kusim` stores two arrays for each direction:

- `is_open`: `True` if the neighbour exists, `False` at the edge,
- `neighbour`: the neighbour's index, or **the block's own index** if there is none (so using it as an index is always safe).

Then the pressure difference "neighbour minus me" is **zero** at a closed edge, and the transmissibility is made zero by multiplying with `is_open`: a no-flow boundary, with **no special case in the formulas**. This is the same trick as the zero padding of Day 9.

A mask and `np.where` (Day 8) make it easy:

```python
moved = i + 1                                  # position after stepping in +x
is_open = (moved >= 0) & (moved < nx)          # still inside the grid?
neighbour = np.where(is_open, n + 1, n)        # the neighbour, or the block itself
```

**Your turn 5.1.** Write `build_neighbours(nx, ny, nz)` that returns two dictionaries, `neighbour` and `is_open`, each with the six keys `"x_minus"`, `"x_plus"`, `"y_minus"`, `"y_plus"`, `"z_minus"`, `"z_plus"`. Use `cell_positions` and the table above. A loop **over the six directions** is fine, but no loop over blocks. Store the offsets and sizes in dictionaries to avoid six copies of the same code.

In [ ]:
DIRECTIONS = ("x_minus", "x_plus", "y_minus", "y_plus", "z_minus", "z_plus")

# YOUR CODE HERE
DIRECTIONS = ("x_minus", "x_plus", "y_minus", "y_plus", "z_minus", "z_plus")

def build_neighbours(nx, ny, nz):
    """Neighbour index and is-open mask of every block, for each of the six directions."""
    pass

In [ ]:
# Check (do not edit)
neighbour, is_open = build_neighbours(4, 3, 2)
assert set(neighbour) == set(DIRECTIONS) and set(is_open) == set(DIRECTIONS), "six keys expected"
assert list(neighbour["x_plus"][[0, 3, 5, 11, 12, 23]]) == [1, 3, 6, 11, 13, 23], "x_plus is wrong"
assert list(neighbour["y_minus"][[0, 3, 5, 11, 12, 23]]) == [0, 3, 1, 7, 12, 19], f"y_minus is {neighbour['y_minus'][[0, 3, 5, 11, 12, 23]]}"
assert list(neighbour["z_plus"][[0, 3, 5, 11, 12, 23]]) == [12, 15, 17, 23, 12, 23], "z_plus is wrong"
assert not is_open["z_minus"][:12].any() and is_open["z_minus"][12:].all(), "only the second layer has a block above"
assert is_open["x_plus"].sum() == 18, "18 blocks have a right-hand neighbour (3 per row x 3 rows x 2 layers)"
print("Correct: the edge blocks point to themselves")

<details><summary>Solution</summary>

```python
DIRECTIONS = ("x_minus", "x_plus", "y_minus", "y_plus", "z_minus", "z_plus")

def build_neighbours(nx, ny, nz):
    """Neighbour index and is-open mask of every block, for each of the six directions."""
    i, j, k = cell_positions(nx, ny, nz)
    n = np.arange(nx * ny * nz)
    step = {"x": 1, "y": nx, "z": nx * ny}                 # index change for one step
    position = {"x": i, "y": j, "z": k}
    size = {"x": nx, "y": ny, "z": nz}
    neighbour, is_open = {}, {}
    for direction in DIRECTIONS:
        axis = direction[0]
        sign = -1 if direction.endswith("minus") else 1
        moved = position[axis] + sign
        is_open[direction] = (moved >= 0) & (moved < size[axis])
        neighbour[direction] = np.where(is_open[direction], n + sign * step[axis], n)
    return neighbour, is_open
```
Compare with `_build_neighbours` in `src/kusim/grid.py`: it is the same idea, written once for all six directions with the `step`, `position` and `size` dictionaries.
</details>

**Your turn 5.2.** Every pair of neighbouring blocks is a **connection** (an interface where fluid can flow). Count each connection once, by counting the open faces in the three **plus** directions only. Store `n_connections` for the 4 × 3 × 2 grid, and compare with the formula $(n_x-1)n_yn_z+n_x(n_y-1)n_z+n_xn_y(n_z-1)$ in `n_formula`.

In [ ]:
neighbour, is_open = build_neighbours(4, 3, 2)

# YOUR CODE HERE
n_connections = None
n_formula = None

In [ ]:
# Check (do not edit)
assert n_connections == 46, f"the 4 x 3 x 2 grid has 46 connections, got {n_connections}"
assert n_formula == 46, f"the formula should also give 46, got {n_formula}"
print("Correct: 18 connections along x, 16 along y, 12 along z")

<details><summary>Solution</summary>

```python
n_connections = sum(is_open[d].sum() for d in ("x_plus", "y_plus", "z_plus"))
n_formula = (4 - 1) * 3 * 2 + 4 * (3 - 1) * 2 + 4 * 3 * (2 - 1)
```
Each connection gets one transmissibility, so the number of connections is the number of non-zero off-diagonal entries (counted twice, once for each side) that the pressure matrix of Day 11 will have.
</details>

## 6. Block sizes, volumes and depths

`kusim` takes the grid sizes the way the original program did: **one `dx` per column**, **one `dy` per row** and **one `dz` per layer** (lists of length `nx`, `ny`, `nz`). Each block then gets its own sizes by looking up its position, with *an array as the index*:

```python
dx_block = dx[i]     # i is the array of column positions of all blocks
```

Using an array as an index returns one item for each entry of the array. This is the NumPy "fancy indexing" and it is how the three size lists become three arrays of length `n_cells`.

The **bulk volume** of a block is $\Delta x\,\Delta y\,\Delta z$ (ft³), the **pore volume** is bulk volume $\times\,\phi$, and in barrels it is divided by 5.615 (Day 4).

**The depth of a block's centre** below the datum: the top of the reservoir, plus the thickness of all the layers above, plus half of the block's own thickness:

$$\text{depth} = \text{top} + \sum_{\text{layers above}} \Delta z + \frac{\Delta z}{2}$$

The sum of the thicknesses above is a `np.cumsum` with a 0 in front (Day 9), taken **one item early**: `np.concatenate(([0.0], np.cumsum(dz)[:-1]))`. Depth matters for the gravity term that we meet later.

**Your turn 6.1.** For the grid with `dx`, `dy`, `dz` below, use `cell_positions` to calculate the per-block arrays `dx_b`, `dy_b`, `dz_b` and the bulk `volume` (ft³). Layer porosities are `[0.20, 0.15]`: calculate `pore_volume_bbl` (array, bbl; per-block porosity from `np.repeat`, as in 3.1). Store `total_pv` (sum of pore volumes) and the original oil in place `ooip_stb` with $S_o=0.8$ everywhere and $B_o=1.2$ rb/STB (Day 4).

In [ ]:
dx = np.array([100.0, 100.0, 150.0, 150.0])
dy = np.array([100.0, 200.0, 100.0])
dz = np.array([20.0, 10.0])
phi_layer = np.array([0.20, 0.15])
i, j, k = cell_positions(4, 3, 2)

# YOUR CODE HERE
dx_b = None
dy_b = None
dz_b = None
volume = None
pore_volume_bbl = None
total_pv = None
ooip_stb = None

In [ ]:
# Check (do not edit)
assert len(volume) == 24 and volume[0] == 200000.0 and volume[2] == 300000.0, f"volume starts {volume[:4]}"
assert abs(volume.sum() - 6_000_000) < 1, f"total bulk volume should be 6,000,000 ft3, got {volume.sum()}"
assert abs(total_pv - 195903.8) < 1, f"total pore volume should be about 195,904 bbl, got {total_pv}"
assert abs(ooip_stb - 130602.6) < 1, f"OOIP should be about 130,603 STB, got {ooip_stb}"
print(f"Correct: bulk volume {volume.sum():,.0f} ft3, pore volume {total_pv:,.0f} bbl, OOIP {ooip_stb:,.0f} STB")

<details><summary>Solution</summary>

```python
dx_b = dx[i]
dy_b = dy[j]
dz_b = dz[k]
volume = dx_b * dy_b * dz_b
phi = np.repeat(phi_layer, 4 * 3)
pore_volume_bbl = volume * phi / 5.615
total_pv = pore_volume_bbl.sum()
ooip_stb = (pore_volume_bbl * 0.8 / 1.2).sum()
```
Indexing `dx[i]` with the array `i` is the line `self.dx = dx[self.i]` in `Grid.__init__`.
</details>

**Your turn 6.2.** The top of the reservoir is at 5000 ft depth (flat). Calculate the array `depth` of the centre depth of every block, using `dz`, `k` and the cumulative-sum idea above. Layer 1 centres should lie at 5010 ft and layer 2 centres at 5025 ft.

In [ ]:
top_depth = 5000.0

# YOUR CODE HERE
depth = None

In [ ]:
# Check (do not edit)
assert np.allclose(depth[:12], 5010.0), "layer 1 centres: 5000 + 20/2 = 5010 ft"
assert np.allclose(depth[12:], 5025.0), "layer 2 centres: 5000 + 20 + 10/2 = 5025 ft"
print("Correct: centre depths are", sorted(set(depth.tolist())))

<details><summary>Solution</summary>

```python
thickness_above = np.concatenate(([0.0], np.cumsum(dz)[:-1]))
depth = top_depth + thickness_above[k] + dz[k] / 2
```
This is the line `self.depth = top[...] + thickness_above[self.k] + self.dz / 2` in `Grid.__init__`.
</details>

## 7. Week 2 project: your own `Grid`

You now have every part of a grid. Put them together.

**Your turn 7.1.** Write `make_grid(dx, dy, dz, top_depth)` returning a dictionary with the keys

`"nx"`, `"ny"`, `"nz"`, `"n_cells"`, `"i"`, `"j"`, `"k"`, `"dx"`, `"dy"`, `"dz"` (per-block arrays), `"volume"`, `"depth"`, `"neighbour"` and `"is_open"` (the two dictionaries of Section 5).

Reuse the functions `cell_positions` and `build_neighbours` from above.

Write it in the cell below.

In [ ]:
# YOUR CODE HERE
def make_grid(dx, dy, dz, top_depth):
    """A grid as a dictionary of arrays, like kusim's Grid (positions counted from 0)."""
    pass

In [ ]:
# Check (do not edit)
my_grid = make_grid([100, 100, 150, 150], [100, 200, 100], [20, 10], 5000.0)
assert my_grid["n_cells"] == 24 and (my_grid["nx"], my_grid["ny"], my_grid["nz"]) == (4, 3, 2)
assert abs(my_grid["volume"].sum() - 6_000_000) < 1, "bulk volume should be 6,000,000 ft3"
assert list(my_grid["depth"][[0, 11, 12, 23]]) == [5010.0, 5010.0, 5025.0, 5025.0], "depths of the first and last blocks of each layer"
assert list(my_grid["neighbour"]["x_plus"][[0, 3, 5, 11, 12, 23]]) == [1, 3, 6, 11, 13, 23], "neighbour x_plus"
assert my_grid["is_open"]["y_plus"].sum() == 16 and my_grid["is_open"]["z_plus"].sum() == 12, "open faces in y_plus and z_plus"
print("Correct: your grid agrees with the reference values of kusim's Grid")

<details><summary>Solution</summary>

```python
def make_grid(dx, dy, dz, top_depth):
    """A grid as a dictionary of arrays, like kusim's Grid (positions counted from 0)."""
    dx, dy, dz = np.asarray(dx, float), np.asarray(dy, float), np.asarray(dz, float)
    nx, ny, nz = len(dx), len(dy), len(dz)
    i, j, k = cell_positions(nx, ny, nz)
    thickness_above = np.concatenate(([0.0], np.cumsum(dz)[:-1]))
    neighbour, is_open = build_neighbours(nx, ny, nz)
    return {
        "nx": nx, "ny": ny, "nz": nz, "n_cells": nx * ny * nz,
        "i": i, "j": j, "k": k,
        "dx": dx[i], "dy": dy[j], "dz": dz[k],
        "volume": dx[i] * dy[j] * dz[k],
        "depth": top_depth + thickness_above[k] + dz[k] / 2,
        "neighbour": neighbour, "is_open": is_open,
    }
```

</details>

### Compare with `kusim`

If the `kusim` package is installed (from Day 11 it will be), the next cell compares *every array* of your grid with `kusim.grid.Grid`. If it is not installed, the cell just says so; the check above already used reference values taken from it.

In [ ]:
try:
    from kusim.grid import Grid
except ImportError:
    print("kusim is not installed here; skipping the comparison (see Day 11).")
else:
    ref = Grid([100, 100, 150, 150], [100, 200, 100], [20, 10], top_depth=5000.0)
    mine = make_grid([100, 100, 150, 150], [100, 200, 100], [20, 10], 5000.0)
    same = True
    for name in ["i", "j", "k", "dx", "dy", "dz", "volume", "depth"]:
        ok = np.allclose(mine[name], getattr(ref, name))
        same &= ok
        print(f"{name:8s}", "same" if ok else "DIFFERENT")
    for d in DIRECTIONS:
        ok = np.array_equal(mine["neighbour"][d], ref.neighbour[d]) and np.array_equal(mine["is_open"][d], ref.is_open[d])
        same &= ok
        print(f"{d:8s}", "same" if ok else "DIFFERENT")
    print("Your grid is identical to kusim's." if same else "Some arrays differ: compare them one by one.")

### Using the grid: Day 9 again

The real test of a grid is to use it. On Day 9 you found the net flow into each block of a row of six by padding the interface flows with zeros. A simulator does it with the **neighbour arrays**: the flow into a block is the sum, over its directions, of *transmissibility $\times$ (neighbour pressure $-$ own pressure)*, and closed edges are automatically zero.

For one row of blocks the two directions are `x_minus` and `x_plus`. With equal block size $\Delta x$ and area $A=\Delta y\,\Delta z$:

$$\text{net inflow}_n=\sum_{d\in\{x-,\,x+\}} T_d\,\bigl(p_{\text{nb}(d)}-p_n\bigr),\qquad T_d=0.001127\,\frac{A\,k_{d}}{\mu\,\Delta x}\,\text{is\_open}_d$$

where $k_d=2k_nk_{\text{nb}}/(k_n+k_{\text{nb}})$ is the harmonic average of the block and its neighbour.

**Your turn 7.2.** Write `net_inflow_x(grid, p, k, mu)` that uses `grid["neighbour"]`, `grid["is_open"]`, `grid["dx"]`, `grid["dy"]` and `grid["dz"]` to return the net inflow into every block, from the `x_minus` and `x_plus` directions only. Then run it on the six-block row of Day 9 (block size 200 × 100 × 20 ft, so $A=2000$ ft²) with the same permeabilities and pressures. The result must equal Day 9's `net6`.

In [ ]:
row = make_grid([200.0] * 6, [100.0], [20.0], 5000.0)
k6 = np.array([100.0, 100.0, 10.0, 50.0, 100.0, 100.0])
p6 = np.linspace(3000, 2900, 6)

# YOUR CODE HERE
def net_inflow_x(grid, p, k, mu):
    """Net flow (bbl/day) into every block from its x neighbours."""
    pass

net_row = None

In [ ]:
# Check (do not edit)
net_row = net_inflow_x(row, p6, k6, mu=1.0)
assert np.allclose(net_row, [-22.54, 18.44, 0.34, -11.27, -7.51, 22.54], atol=0.01), f"net_row is {np.round(net_row, 2)}"
assert abs(net_row.sum()) < 1e-9, "the closed ends make the total zero"
print("Correct: the same numbers as Day 9, but found from the neighbour arrays:", np.round(net_row, 2))

<details><summary>Solution</summary>

```python
def net_inflow_x(grid, p, k, mu):
    """Net flow (bbl/day) into every block from its x neighbours."""
    area = grid["dy"] * grid["dz"]
    net = np.zeros(grid["n_cells"])
    for d in ("x_minus", "x_plus"):
        nb = grid["neighbour"][d]
        k_face = 2 * k * k[nb] / (k + k[nb])
        trans = DARCY_UNITS * area * k_face / (mu * grid["dx"]) * grid["is_open"][d]
        net += trans * (p[nb] - p)
    return net

net_row = net_inflow_x(row, p6, k6, mu=1.0)
```
Day 9 needed a special padding for the ends. Here the ends need **nothing**: the first block's `x_minus` neighbour is itself, the pressure difference is zero, and `is_open` is also zero. The same function works unchanged for a 2-D or 3-D grid by looping over all six directions, with the right area and size for each. This is how `simulator.py` assembles the flow terms.
</details>

## 8. Week 2 review quiz

Answer in the comments, then open the answers. The questions cover Days 6 to 10.

1. **Day 6.** `corey = {"swc": 0.2, "nw": 2.0}`: how do you read `nw`, and how do you add the key `"sor"`?
2. **Day 7.** Why is `2.5 * arr + 1` preferred to a loop for a large array?
3. **Day 7.** A capillary pressure of 15 psi, oil gravity 0.85 and water 1.0: how high above the free-water level? (Equation: $h=P_{cow}/(0.433(\gamma_w-\gamma_o))$.)
4. **Day 8.** What do `np.clip(s, 0, 1)` and `np.where(krw > 0, a, b)` do, and where does the simulator use `np.clip`?
5. **Day 8.** What is the fractional flow of water where $k_{rw}=0$, and where $k_{ro}=0$?
6. **Day 9.** The pressure drop across a block is doubled and the oil viscosity is halved. What happens to the Darcy rate?
7. **Day 10.** In a 4 × 3 × 2 grid, what is the block number of $(i,j,k)=(2,3,1)$?
8. **Day 10.** In the same grid, what are the $(i,j,k)$ of block 17, and which of its six neighbours do not exist?

In [ ]:
# 1.
# 2.
# 3.
# 4.
# 5.
# 6.
# 7.
# 8.

<details><summary>Answers</summary>

1. `corey["nw"]` reads it (the key goes in square brackets); `corey["sor"] = 0.2` adds the key.
2. The loop runs one Python step at a time; the array operation runs in compiled code on all items at once. It is faster and reads like the equation.
3. $h = 15/(0.433\times0.15) = 230.9$ ft.
4. `np.clip(s, 0, 1)` replaces values below 0 by 0 and above 1 by 1; `np.where(cond, a, b)` takes `a` where the condition holds and `b` elsewhere (an `if` for every item). The simulator's explicit saturation update ends with `np.clip(s_new, swc, 1 - sor)`.
5. $f_w=0$ where $k_{rw}=0$ (no mobile water) and $f_w=1$ where $k_{ro}=0$ (no mobile oil).
6. It is multiplied by 4: twice from the pressure drop and twice from the halved viscosity, as $q\propto \Delta P/\mu$.
7. $(k-1)\,n_xn_y+(j-1)\,n_x+i = 0+2\cdot4+2 = 10$.
8. Index $16$: $k_0=16//12=1$, remainder 4, so $j_0=4//4=1$ and $i_0=4\bmod4=0$: $(i,j,k)=(1,2,2)$. It is in the first column (no `x_minus`) and in the bottom layer (no `z_plus`). Its neighbours that exist: `x_plus` (18), `y_minus` (13), `y_plus` (21) and `z_minus` (5).
</details>

## 9. Pocket reference (today's Python and formulas)

| You want to... | Write |
|---|---|
| block number from $(i,j,k)$ (from 1) | `(k - 1) * nx * ny + (j - 1) * nx + i` |
| position from the index $n$ (from 0) | `i = n % nx`, `j = (n // nx) % ny`, `k = n // (nx * ny)` |
| integer division / remainder | `a // b`, `a % b` (also on arrays) |
| flat array to layers | `a.reshape(nz, ny, nx)` |
| back to flat | `a3d.ravel()` |
| one layer / one column | `a3d[k]`, `a3d[:, j, i]` |
| average over layers / of each layer | `a3d.mean(axis=0)`, `a3d.mean(axis=(1, 2))` |
| repeat each value | `np.repeat([a, b], m)` |
| per-block value from a per-column list | `dx[i]` (an array as an index) |
| neighbour or itself | `np.where(is_open, n + step, n)` |
| count open faces | `is_open.sum()` |

## 10. Wrap-up

Today you:

- numbered blocks in 1-D, 2-D and 3-D, and met the off-by-one trap,
- viewed flat arrays as layers with `reshape`, and averaged with `axis`,
- converted between block numbers and `(i, j, k)` with `//` and `%`, on all blocks at once,
- built the neighbour arrays and the closed-boundary masks of a grid,
- made block volumes, pore volumes, OOIP and depths from the per-column, per-row and per-layer sizes,
- assembled your own `Grid`, and reproduced Day 9's flow balance from its neighbour arrays.

**Week 2 is done.** You can now load rock-fluid tables, make the Corey and fractional-flow curves with NumPy, calculate Darcy flow, and describe a 3-D grid with its neighbours: all the ingredients of a simulator's pressure equation.

**Next, Day 11 (Week 3):** the pressure equation. You will turn the flow balance of each block into one equation per block, assemble the **matrix** for a 1-D steady problem with a loop, and solve it with `np.linalg.solve`. From Day 11 on, a setup cell installs the `kusim` package.